In [1]:
# ============================================================
# ERIP - ENTERPRISE RISK INTELLIGENCE PLATFORM
# ============================================================
#
# Notebook
# --------
# nb_build_rating_dimension
#
# Layer
# -----
# Silver Layer
#
# Purpose
# -------
# Build the enterprise Rating Dimension from the Bronze
# Internal Rating Engine table.
#
# Business Objective
# ------------------
# Create a standardized, analytics-ready credit rating entity
# that supports:
#
# • Probability of Default (PD)
# • Loss Given Default (LGD)
# • Exposure at Default (EAD)
# • Expected Credit Loss (ECL)
# • IFRS 9 Stage Recommendation
# • Model Governance
# • Executive Risk Analytics
#
# Enterprise Concepts
# -------------------
# ✓ Medallion Architecture
# ✓ Credit Risk Analytics
# ✓ Rating Model Governance
# ✓ IFRS 9
# ✓ Expected Credit Loss Foundation
# ✓ Analytics Engineering
# ============================================================

from pyspark.sql.functions import *
from pyspark.sql.window import Window
from datetime import datetime

# ------------------------------------------------------------
# SECTION 1 - Source / Target Configuration
# ------------------------------------------------------------

source_table = "bronze_internal_rating_engine"
target_table = "silver_rating"
pipeline_name = "nb_build_rating_dimension"

# Pipeline Execution Timestamp
run_start_time = datetime.now()

print("ERIP Silver Rating Dimension Build Started")


StatementMeta(, d313ad61-d8b7-4a3d-9623-8b9d814c2591, 3, Finished, Available, Finished, False)

ERIP Silver Rating Dimension Build Started


In [2]:
# ============================================================
# SECTION 2 - READ BRONZE INTERNAL RATING TABLE
# ============================================================
#
# Purpose
# -------
# Read the validated/governed Bronze Delta table.
#
# Enterprise Concepts
# -------------------
# ✓ Delta Lake
# ✓ Data Lineage
# ✓ Medallion Architecture
# ============================================================

rating_bronze_df = spark.table(source_table)

display(rating_bronze_df.limit(10))

print(f"Rows read : {rating_bronze_df.count()}")
print(f"Columns   : {len(rating_bronze_df.columns)}")

StatementMeta(, d313ad61-d8b7-4a3d-9623-8b9d814c2591, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f65baafe-dc8f-45c3-a2ee-fc42c3750101)

Rows read : 1000
Columns   : 22


In [3]:
# ============================================================
# SECTION 3 - STANDARDIZE RATING DIMENSION
# ============================================================
#
# Purpose
# -------
# Standardize credit rating, model, and risk attributes into
# a business-ready Silver Rating entity.
#
# Standardization Activities
# --------------------------
# • Trim whitespace
# • Standardize text case
# • Convert PD, LGD, EAD and Expected Loss to numeric values
# • Convert rating dates to date type
# • Remove duplicate rating records
#
# Enterprise Concepts
# -------------------
# ✓ Data Standardization
# ✓ Credit Risk Entity Modelling
# ✓ Model Governance
# ✓ Risk Analytics Foundation
# ============================================================

silver_rating_df = (
    rating_bronze_df
    .select(
        upper(trim(col("rating_record_id"))).alias("rating_record_id"),
        upper(trim(col("customer_id"))).alias("customer_id"),
        to_date(col("rating_date")).alias("rating_date"),
        upper(trim(col("previous_internal_grade"))).alias("previous_internal_grade"),
        upper(trim(col("current_internal_grade"))).alias("current_internal_grade"),
        col("pd").cast("double").alias("pd"),
        initcap(trim(col("pd_band"))).alias("pd_band"),
        col("lgd").cast("double").alias("lgd"),
        col("ead").cast("double").alias("ead"),
        col("expected_loss").cast("double").alias("expected_loss"),
        upper(trim(col("watchlist_flag"))).alias("watchlist_flag"),
        initcap(trim(col("scorecard_type"))).alias("scorecard_type"),
        col("scorecard_score").cast("int").alias("scorecard_score"),
        upper(trim(col("model_version"))).alias("model_version"),
        upper(trim(col("model_override_flag"))).alias("model_override_flag"),
        initcap(trim(col("override_reason"))).alias("override_reason"),
        initcap(trim(col("ifrs9_stage_recommendation"))).alias("ifrs9_stage_recommendation"),
        current_timestamp().alias("silver_updated_timestamp")
    )
    .dropDuplicates(["rating_record_id"])
)

StatementMeta(, d313ad61-d8b7-4a3d-9623-8b9d814c2591, 5, Finished, Available, Finished, False)

In [4]:
# ============================================================
# SECTION 4 - BUSINESS ENRICHMENT
# ============================================================
#
# Purpose
# -------
# Create derived business attributes required for enterprise
# credit risk analytics and model governance.
#
# Derived Attributes
# ------------------
# • Rating Surrogate Key
# • Rating Risk Category
# • Watchlist Indicator
# • Override Indicator
# • IFRS 9 Numeric Stage
# • ECL Recalculation Check
#
# Enterprise Concepts
# -------------------
# ✓ IFRS 9
# ✓ Expected Credit Loss
# ✓ Credit Risk Scoring
# ✓ Model Override Governance
# ✓ Dimensional Modelling
# ============================================================

window_spec = Window.orderBy("rating_record_id")

silver_rating_df = (
    silver_rating_df
    .withColumn("rating_sk", row_number().over(window_spec))
    .withColumn(
        "rating_risk_category",
        when(col("current_internal_grade").isin("IG1", "IG2", "IG3"), "Low Risk")
        .when(col("current_internal_grade").isin("IG4", "NIG1"), "Medium Risk")
        .when(col("current_internal_grade").isin("NIG2", "NIG3"), "High Risk")
        .when(col("current_internal_grade") == "DEFAULT", "Default")
        .otherwise("Unknown")
    )
    .withColumn(
        "is_watchlist",
        when(col("watchlist_flag") == "Y", lit(1)).otherwise(lit(0))
    )
    .withColumn(
        "is_model_override",
        when(col("model_override_flag") == "Y", lit(1)).otherwise(lit(0))
    )
    .withColumn(
        "ifrs9_stage_numeric",
        regexp_extract(col("ifrs9_stage_recommendation"), r"(\d+)", 1).cast("int")
    )
    .withColumn(
        "expected_loss_recalculated",
        col("pd") * col("lgd") * col("ead")
    )
    .withColumn(
        "expected_loss_variance",
        col("expected_loss_recalculated") - col("expected_loss")
    )
    .select(
        "rating_sk",
        "rating_record_id",
        "customer_id",
        "rating_date",
        "previous_internal_grade",
        "current_internal_grade",
        "rating_risk_category",
        "pd",
        "pd_band",
        "lgd",
        "ead",
        "expected_loss",
        "expected_loss_recalculated",
        "expected_loss_variance",
        "watchlist_flag",
        "is_watchlist",
        "scorecard_type",
        "scorecard_score",
        "model_version",
        "model_override_flag",
        "is_model_override",
        "override_reason",
        "ifrs9_stage_recommendation",
        "ifrs9_stage_numeric",
        "silver_updated_timestamp"
    )
)

display(silver_rating_df.limit(10))

StatementMeta(, d313ad61-d8b7-4a3d-9623-8b9d814c2591, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 135d4d43-ac37-4d38-953b-93c900ed5497)

In [5]:
# ============================================================
# SECTION 5 - SILVER DATA QUALITY VALIDATION
# ============================================================
#
# Purpose
# -------
# Validate the Silver Rating Dimension before publishing.
#
# Validation Checks
# -----------------
# • Duplicate Rating Record IDs
# • Null Rating Record IDs
# • Null Customer IDs
# • Null Surrogate Keys
# • Invalid PD values
# • Invalid LGD values
# • Invalid EAD values
# • Invalid IFRS 9 stages
#
# Enterprise Concepts
# -------------------
# ✓ Data Quality
# ✓ Credit Risk Governance
# ✓ Model Risk Controls
# ✓ Trusted Analytics Layer
# ============================================================

total_rows = silver_rating_df.count()

duplicate_rating_ids = total_rows - silver_rating_df.select("rating_record_id").distinct().count()

null_rating_ids = silver_rating_df.filter(
    col("rating_record_id").isNull()
).count()

null_customer_ids = silver_rating_df.filter(
    col("customer_id").isNull()
).count()

null_surrogate_keys = silver_rating_df.filter(
    col("rating_sk").isNull()
).count()

invalid_pd = silver_rating_df.filter(
    (col("pd") < 0) | (col("pd") > 1)
).count()

invalid_lgd = silver_rating_df.filter(
    (col("lgd") < 0) | (col("lgd") > 1)
).count()

invalid_ead = silver_rating_df.filter(
    col("ead") < 0
).count()

invalid_ifrs9_stage = silver_rating_df.filter(
    ~col("ifrs9_stage_numeric").isin(1, 2, 3)
).count()

print("Silver Rating Quality Checks")
print("----------------------------")
print(f"Rows: {total_rows}")
print(f"Duplicate Rating IDs: {duplicate_rating_ids}")
print(f"Null Rating IDs: {null_rating_ids}")
print(f"Null Customer IDs: {null_customer_ids}")
print(f"Null Surrogate Keys: {null_surrogate_keys}")
print(f"Invalid PD: {invalid_pd}")
print(f"Invalid LGD: {invalid_lgd}")
print(f"Invalid EAD: {invalid_ead}")
print(f"Invalid IFRS 9 Stage: {invalid_ifrs9_stage}")

if (
    duplicate_rating_ids > 0 or
    null_rating_ids > 0 or
    null_customer_ids > 0 or
    null_surrogate_keys > 0 or
    invalid_pd > 0 or
    invalid_lgd > 0 or
    invalid_ead > 0 or
    invalid_ifrs9_stage > 0
):
    raise Exception("Silver Rating Validation Failed")
else:
    print("✓ Silver Rating Validation Passed")

StatementMeta(, d313ad61-d8b7-4a3d-9623-8b9d814c2591, 7, Finished, Available, Finished, False)

Silver Rating Quality Checks
----------------------------
Rows: 1000
Duplicate Rating IDs: 0
Null Rating IDs: 0
Null Customer IDs: 0
Null Surrogate Keys: 0
Invalid PD: 0
Invalid LGD: 0
Invalid EAD: 0
Invalid IFRS 9 Stage: 0
✓ Silver Rating Validation Passed


In [6]:
# ============================================================
# SECTION 6 - WRITE SILVER DELTA TABLE
# ============================================================
#
# Purpose
# -------
# Publish the enterprise Rating Dimension to the Silver Layer.
#
# Output
# ------
# silver_rating
#
# Enterprise Concepts
# -------------------
# ✓ Delta Lake
# ✓ Credit Risk Analytics
# ✓ Enterprise Data Platform
# ============================================================

silver_rating_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(target_table)

print(f"✓ Silver table created: {target_table}")
print(f"Rows written: {silver_rating_df.count()}")

StatementMeta(, d313ad61-d8b7-4a3d-9623-8b9d814c2591, 8, Finished, Available, Finished, False)

✓ Silver table created: silver_rating
Rows written: 1000
